# <b>Agent Platform AutoML による物体検出</b>

## <b>学習目標</b> ##

1. Agent Platform 上で物体検出用のマネージド データセットを作成する方法を学ぶ
1. AutoML 物体検出モデルをトレーニングする方法を学ぶ
1. AutoML でトレーニングされたモデルを評価する方法を学ぶ
1. AutoML でトレーニングされたモデルをエンドポイントにデプロイする方法を学ぶ
1. デプロイされたモデルを使用して新しいデータで予測を行う方法を学ぶ

このノートブックでは、Agent Platform を使用して、指定された画像内の複数の物体を検出し、画像内の物体とその位置に関する情報を提供できる AutoML モデルをトレーニングします。

まず、Agent Platform 上でマネージド データセットを作成し、一般公開されている画像セットをそこにインポートします。その後、AutoML モデルをトレーニング、評価し、エンドポイントにデプロイします。最後に、モデルに予測リクエストを送信します。

### セットアップ

In [1]:
import os
from datetime import datetime

from google.cloud import aiplatform

PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
BUCKET = PROJECT  # デフォルトは PROJECT
REGION = "us-central1"
TIMESTAMP = datetime.now().strftime("%Y%m%d%H%M%S")

os.environ["BUCKET"] = BUCKET
os.environ["PROJECT"] = PROJECT
os.environ["REGION"] = REGION

aiplatform.init(project=PROJECT, location=REGION)


## ステップ 1: トレーニングデータの準備とフォーマット

Agent Platform でマネージド データセットを作成する最初のステップは、トレーニングデータを準備することです。このケースでは、トレーニングデータセットは画像と、画像内の物体の位置（境界ボックス / バウンディングボックスの座標による）および物体の種類（ラベルによる）を特定する情報で構成されます。
物体検出用のマネージド データセットを準備する際の制約事項と一般的なルールは次のとおりです:

* 次の画像形式がサポートされています: JPEG、PNG、GIF、BMP、ICO。画像あたりの最大ファイルサイズは 30MB です。

* ラベル（つまり画像内で検出したい物体の種類）あたり約 1,000 枚のトレーニング画像を用意することが推奨されます。各ラベルには少なくとも 10 枚の画像が必要で、それぞれに少なくとも 1 つのアノテーション（境界ボックスとラベル）が必要です。一般的に、ラベルあたりの画像数が多いほど、モデルのパフォーマンスが向上します。

### 画像 URI とラベルを含む CSV ファイルの作成

画像ファイルが Cloud Storage バケットにアップロードされたら、画像 URI、境界ボックス情報、オブジェクトラベルを一覧にした CSV（または JSONL）ファイルを作成する必要があります。CSV ファイルには画像内の境界ボックスごとに 1 行が含まれるため、2 つの境界ボックスを持つ画像の場合、同じ画像 URI を共有する 2 つの行が CSV ファイルに対応して存在することになります。

下の例では、行 1 と行 2 は、`(car,0.1,0.1,,,0.3,0.3,,)` と `(bike,.7,.6,,,.8,.9,,)` という 2 つのアノテーションを持つ同じ画像を参照しています。アノテーションの最初の要素は境界ボックス内のオブジェクトラベルであり、残りは画像内の境界ボックスの座標です（詳細は後述）。


行 3 はアノテーションが 1 つだけ `(car,0.1,0.1,0.2,0.1,0.2,0.3,0.1,0.3)` ある画像を参照し、行 4 はアノテーションのない画像を参照しています。

最初の列はデータ分割に対応し（これはオプションです。指定しない場合、AutoML は 80/10/10 の分割を行います）、2 番目の列は画像 URI、最後の列がアノテーションを保持します。

境界ボックスは次の 2 つの形式のいずれかにできます:
- 2 つの頂点（長方形の対角線上にある反対側の点の x, y 座標）を含む: `(x_min, y_min, , ,x_max,y_max,,)` または
- 4 つの頂点すべてを含む: `(x_min,y_min,x_max,y_min,x_max,y_max,x_min,y_max)`
**例:**

```bash
TRAIN,gs://folder/image1.png,car,0.1,0.1,,,0.3,0.3,,
TRAIN,gs://folder/image1.png,bike,.7,.6,,,.8,.9,,
UNASSIGNED,gs://folder/im2.png,car,0.1,0.1,0.2,0.1,0.2,0.3,0.1,0.3
TEST,gs://folder/im3.png,,,,,,,,,
```

上記の各行には次の列があります:

1. <b>行内のコンテンツが割り当てられるデータセット。</b> - `TRAIN`、`VALIDATE`、`TEST`、または `UNASSIGNED`
1. <b>アノテーション対象のコンテンツ。</b> - 画像の GCS URI が含まれます
1. <b>オブジェクトがどのように分類されるかを識別するラベル。</b>
1. <b>画像内のオブジェクトの境界ボックス。</b>

このラボでは、あらかじめ作成されている CSV を使用します。CSV 内の画像 URI は GCS 内の一般公開されている画像を指しており、各画像内の食品／食材に関連するアノテーションが含まれています。

GCS バケットが存在することを確認し（存在しない場合は作成）、CSV を GCS バケットにコピーします。

In [ ]:
%%bash
exists=$(gcloud storage ls --buckets | grep -w gs://${BUCKET}/)

if [ -n "$exists" ]; then
   echo -e "Bucket gs://${BUCKET} already exists."
    
else
   echo "Creating a new GCS bucket."
   gcloud storage buckets create --location=${REGION} gs://${BUCKET}
   echo -e "\nHere are your current buckets:"
   gcloud storage ls
fi

CSV ファイルの最初の数行を確認し、GCS バケットにコピーします。

In [ ]:
!head ../dataset_import_files/salads.csv

In [ ]:
!gcloud storage cp ../dataset_import_files/salads.csv gs://{BUCKET}/

## ステップ 2: マネージド データセットの作成

次のステップは、Agent Platform 上で物体検出用のマネージド データセットを作成することです。
<br/>
**注**: これには 5〜10 分かかる場合があります。

In [ ]:
DATASET_NAME = "salad_dataset"

ds = aiplatform.ImageDataset.create(
    display_name=DATASET_NAME,
    gcs_source=f"gs://{BUCKET}/salads.csv",
    import_schema_uri=aiplatform.schema.dataset.ioformat.image.bounding_box,
)

## ステップ 3: AutoML モデルのトレーニング
トレーニング ジョブを起動します。`job.run()` に次のパラメータを設定します:
- `dataset`: 先ほど作成したマネージド データセット
- `model_display_name`: モデルの表示名。トレーニング完了後、トレーニング済みモデルは Agent Platform Model オブジェクトになります。
- `training_fraction_split`: トレーニングに使用するデータの割合。明示的に 0.8（80%）に設定します。
- `validation_fraction_split`: 検証に使用するデータの割合。明示的に 0.1（10%）に設定します。
- `test_fraction_split`: テストに使用するデータの割合。明示的に 0.1（10%）に設定します。
- `budget_milli_node_hours`: トレーニング ジョブのミリノード時間（例: 1000 は 1 ノード時間を意味します）。AutoML 物体検出トレーニング ジョブの最小値は 20000（20 ノード時間）です。
- `disable_early_stopping`: ブール値。明示的に False に設定します（つまり早期打ち切りを有効にします）。

In [ ]:
MODEL_DISPLAY_NAME = "salad"

job = aiplatform.AutoMLImageTrainingJob(
    display_name="salad_" + TIMESTAMP, prediction_type="object_detection"
)

model = job.run(
    # TODO: トレーニング ジョブを起動
    # トレーニング分割を 80%、検証とテストをそれぞれ 10% に設定
    # 予算を 20 ノード時間に設定
    # 早期打ち切り（Early Stopping）を有効化
    model_display_name=MODEL_DISPLAY_NAME,
)


**注:** モデルのトレーニングには 1.5〜2.5 時間かかる場合があります。先に進む前に、モデルのトレーニングが完了するのを待つ必要があります。

モデルリソースを取得し、評価指標を取得します。ここでは [Mean Average Precision](https://blog.paperspace.com/mean-average-precision/)（mAP）を確認します。物体検出モデルの評価指標の詳細については、[こちらのリソース](https://pyimagesearch.com/2016/11/07/intersection-over-union-iou-for-object-detection/)を参照してください。

In [ ]:
# 表示名でモデルリソースを取得
models = aiplatform.Model.list(filter="display_name=salad")
salad_model = models[0]

# モデルの評価を取得
client_options = {"api_endpoint": f"{REGION}-aiplatform.googleapis.com"}
model_service_client = aiplatform.gapic.ModelServiceClient(
    client_options=client_options
)

model_evaluations = model_service_client.list_model_evaluations(
    parent=salad_model.resource_name
)
evals = list(model_evaluations)[0]

for metric in list(evals.metrics["boundingBoxMetrics"]):
    try:
        print(f"Mean Average Precision (mAP): {metric['meanAveragePrecision']}")
    except KeyError:
        print("IOU threshold is too high to detect positive cases.")

    print(f"Intersection over union (IoU): {metric['iouThreshold']}\n")

## ステップ 6: モデルのデプロイ

トレーニングされたモデルのパフォーマンスに満足したら、API 経由で予測を利用できるようにデプロイします。

In [ ]:
ENDPOINT_DISPLAY_NAME='salad_model_endpoint'

# TODO: モデルをエンドポイントにデプロイ
endpoint =


## ステップ 7: 予測リクエストの送信

この例では、プロジェクトの Cloud Storage バケットに保存されている画像から個別の予測を呼び出します。
物体検出モデルは、入力画像に対して多くの境界ボックスを出力します。出力として、各ボックスに以下が含まれることを想定しています:
1. ラベル
1. 信頼度スコア

予測を取得するためのデータの準備には、次の手順が含まれます:
1. `base64.b64encode` で画像をエンコードしてエンコード済みバイトを取得し、`.decode("utf-8")` を適用してデコード済み文字列を返します。
1. `ImageObjectDetectionPredictionInstance` を作成します
1. `ImageObjectDetectionPredictionParams` を作成します。これは主に信頼度しきい値を設定するために使用されます。
1. 予測リクエストを行います

In [ ]:
import base64

from google.cloud.aiplatform.gapic.schema import predict

TEST_IMAGE = "../test_images/salad.jpg"  # 任意の Cloud Storage バケットにアップロードされた画像に置き換えてください

# ファイルを読み込む
with open(TEST_IMAGE, "rb") as f:
    file_content = f.read()

encoded_content = base64.b64encode(file_content).decode("utf-8")
instance = predict.instance.ImageObjectDetectionPredictionInstance(
    content=encoded_content
).to_value()

parameters = predict.params.ImageObjectDetectionPredictionParams(
    confidence_threshold=0.7
).to_value()

response = endpoint.predict(instances=[instance], parameters=parameters)
preds = response.predictions[0]
preds


デプロイされたモデルからレスポンスオブジェクトを取得できたので、その予測結果（つまり、上のセルで送信した画像からモデルが検出した境界ボックスとオブジェクト）を検査できます:

`preds` は次のキーを持つ Python 辞書です:
* `bboxes`: 予測された境界ボックス座標のリスト
* `displayNames`: 境界ボックスに対して予測されたラベルのリスト
* `confidences`: 予測信頼度のリスト

In [ ]:
from itertools import cycle

import matplotlib.patches as patches
import matplotlib.pyplot as plt
from PIL import Image

# 各ラベルに異なる色を生成
cycol = cycle("bgrcmk")

img = Image.open(TEST_IMAGE)
fig, ax = plt.subplots()
ax.imshow(img)

for i in range(len(preds["displayNames"])):
    bbox = preds["bboxes"][i]
    display_name = preds["displayNames"][i]
    confidence = preds["confidences"][i]
    xy = (int(bbox[0] * img.width), int(bbox[2] * img.height))
    w = int(img.width * (bbox[1] - bbox[0]))
    h = int(img.height * (bbox[3] - bbox[2]))
    r = patches.Rectangle(
        xy,
        w,
        h,
        linewidth=1,
        facecolor="none",
        label=f"{display_name}: {confidence:.3f}",
        edgecolor=next(cycol),
    )
    ax.add_patch(r)

plt.legend()
plt.show()


お疲れさまでした。このラボでは、物体検出用の Agent Platform マネージド データセットを作成しました。その後、AutoML モデルをトレーニングし、そのトレーニング済みモデルをエンドポイントにデプロイして予測を提供できるようにしました。